[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_3_Nonlinear_Charpit_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" style="text-align:right">

# 📡 اللاب 3: المعادلات غير الخطية من الرتبة الأولى، طريقة شاربيت، وهوامش أمان العوائق
### *غرفة التحكم: هامش الأمان الإقليدي حول الصخور ومعادلة المسافة (Eikonal)*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**المرجع في المذكرة:** الباب الأول، الأقسام 1.3.2–1.3.6 — **ترقيم المذكرة ص 16–26** *(صفحات ملف الـ PDF من 17 إلى 27)*

---
### 🗺️ موقعك في سلسلة بناء محرك الملاحة (Rover Pipeline)
```
[اللاب 1: حساب الميول] ──► [اللاب 2: خطوط الانسياب] ──► 📍 [اللاب 3: معادلة المسافة وهوامش الأمان] ──► [اللاب 4–5: نظرية لابلاس] ──► [اللاب 6: محرك الملاحة الكامل]
```

---
### 🧭 حجر رشيد (The Rosetta Stone): الربط بين السبورة ومحرك المركبة

| 1. رمز المذكرة (PDF pp. 17–27) | 2. المعنى الهندسي والفيزيائي | 3. الدالة الفعلية في المشروع (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $f(x, y, z, p, q) = 0$ غير خطية | معادلة تفاضلية تحتوي على $p^2, q^2$ أو $pq$ | حساب المسافات الإقليدية وجبهات الموجات |
| $p^2 + q^2 = 1$ (أو $\|\nabla z\| = 1$) | **معادلة المسافة (Eikonal):** دالة تزداد بمعدل 1 لكل متر نبتعد فيه عن المصدر | [`terrain.dilate(raw, rover_radius)`](rover_path_finder/terrain.py) (توسيع العوائق بنصف قطر المركبة) |
| $z = px + qy + f(p, q)$ | **صيغة كليروت (Clairaut Form):** عائلة مستويات مماسة؛ حلها التام $z = ax + by + f(a,b)$ | تمثيل جبهات الموجات المستوية وزوايا المسح |
| $(x - x_0)^2 + (y - y_0)^2 \le R^2$ | قرص الأمان الدائري حول كل صخرة أو حفرة لحماية هيكل المركبة | `yy**2 + xx**2 <= radius**2` في `terrain.dilate` |

---
## 1. الفكرة ببساطة: كيف تحسب المركبة المسافة الآمنة حول الصخور؟

المركبة الفضائية ليست نقطة هندسية بلا أبعاد، بل لها عرض فعلي (`rover_radius`). إذا خططنا مساراً يمر بمحاذاة حافة حفرة مباشرة، فستسقط عجلات المركبة في الحفرة!

لذلك، قبل حساب حقل الجهد في [`terrain.py`](rover_path_finder/terrain.py)، تقوم الدالة `dilate(raw, radius)` بحساب **خريطة المسافة الإقليدية** حول كل عائق لتوسيعه بمقدار نصف قطر المركبة:
* إذا سمينا المسافة الإقليدية من نقطة الأصل $z(x,y) = \sqrt{x^2 + y^2}$، فإن ميل هذه الدالة في اتجاه $x$ و $y$ هو:
  $$p = \frac{\partial z}{\partial x} = \frac{x}{\sqrt{x^2+y^2}}, \qquad q = \frac{\partial z}{\partial y} = \frac{y}{\sqrt{x^2+y^2}}$$
* بتربيع الطرفين وجمعهما نحصل على أشهر معادلة تفاضلية جزئية غير خطية من الرتبة الأولى (**معادلة Eikonal**):
  $$p^2 + q^2 = 1 \quad \Longleftrightarrow \quad \|\nabla z\| = 1$$

هذه المعادلة **غير خطية** لأن المشتقتين $p$ و $q$ مرفوعتان للأس 2، فلا يمكن حلها بطريقة لاجرانج التي درسناها في اللاب 2! في هذا اللاب سنتعلم كيف نحل المعادلات غير الخطية من الرتبة الأولى باستخدام **الأشكال القياسية الأربعة** و**طريقة شاربيت العامة (Charpit's Method)**.

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab3

%matplotlib inline
print("تم تهيئة بيئة اللاب 3 بنجاح!")


<div dir="rtl" style="text-align:right">

---
## 2. الشرح الكامل للأشكال القياسية الأربعة وطريقة شاربيت العامة (PDF ص 17–27)

عندما تواجه معادلة غير خطية من الرتبة الأولى $f(x, y, z, p, q) = 0$، هدفنا هو إيجاد **التكامل التام (Complete Integral)**، وهو حل يحتوي على **ثابتين اختياريين** $(a, b)$ (أو $a, c$) بعدد المتغيرات المستقلة $(x, y)$.

### 🧭 خريطة القرار السريع: كيف تختار طريقة الحل في 5 ثوانٍ؟

| الشكل القياسي في المذكرة | كيف تميزه بمجرد النظر؟ | خطوات الحل القياسية |
| :--- | :--- | :--- |
| **Form 1 (القسم 1.3.2)** *(PDF ص 17–19)* | $f(p, q) = 0$ (تحتوي على $p, q$ فقط، بلا $x, y, z$) | نضع $p = a, q = b$، ونوجد $b = F_1(a)$ من المعادلة، ونعوض في $z = ax + F_1(a)y + c$ |
| **Form 2 (Clairaut — القسم 1.3.3)** *(PDF ص 19–20)* | $z = px + qy + f(p, q)$ | نضع مباشرةً $p = a$ و $q = b$ فيكون الحل التام: $z = ax + by + f(a, b)$ |
| **Form 3 (القسم 1.3.4)** *(PDF ص 21–23)* | $f(z, p, q) = 0$ (تحتوي على $z, p, q$ فقط، بلا $x, y$) | نفرض $u = x + ay$ ونستبدل $p = \frac{dz}{du}$ و $q = a\frac{dz}{du}$ ونحل المعادلة التفاضلية العادية |
| **Form 4 (Separable — القسم 1.3.5)** *(PDF ص 23–25)* | $f_1(x, p) = f_2(y, q)$ (لا يوجد $z$، ويمكن فصل $x,p$ عن $y,q$) | نساوي الطرفين بثابت $a$: $f_1(x,p)=a, f_2(y,q)=a$، نوجد $p(x), q(y)$ ونكامل $dz = p\,dx + q\,dy$ |
| **طريقة شاربيت العامة (القسم 1.3.6)** *(PDF ص 25–27)* | $f(x, y, z, p, q) = 0$ (جميع المتغيرات مختلطة ولا تنطبق الأشكال 1–4) | نكتب كسور شاربيت الخمسة، نوجد علاقة ثانية بين $p, q$ بثابت $a$، ثم نكامل $dz = p\,dx + q\,dy$ |

---

### 📖 تفصيل الأشكال الأربعة مع أمثلة المذكرة المحلولة خطوة بخطوة:

#### 1️⃣ الشكل الأول $f(p, q) = 0$ (القسم 1.3.2 — ص 16–18 بالمذكرة / PDF ص 17–19):
* **لماذا ينجح؟** إذا كان $p = a$ و $q = b$ ثابتين، فإن التفاضل الكلي $dz = p\,dx + q\,dy = a\,dx + b\,dy$ يعطي بالتكامل $z = ax + by + c$. وبما أن $F(a, b) = 0$، نحسب $b$ بدلالة $a$ ليبقى ثابتان مستقلان فقط $(a, c)$!
* **مثال 1 (PDF ص 18):** حل $p^2 + p = q^2$:
  * نضع $p = a, q = b \implies a^2 + a = b^2 \implies b = \pm\sqrt{a^2 + a}$.
  * إذن التكامل التام هو: $z = ax \pm \sqrt{a^2 + a}\,y + c$.
* **مثال 2 (PDF ص 18–19):** حل $pq = k$ (حيث $k$ ثابت معطى):
  * نضع $p = a, q = b \implies ab = k \implies b = \frac{k}{a}$.
  * إذن التكامل التام هو: $z = ax + \frac{k}{a}y + c$.

---

#### 2️⃣ الشكل الثاني: معادلة كليروت $z = px + qy + f(p, q)$ (القسم 1.3.3 — ص 18–19 بالمذكرة / PDF ص 19–20):
* **قاعدة الحل:** ضع $p = a$ و $q = b$ مباشرةً في المعادلة! **(تنبيه: لا تضف ثابتاً ثالثاً $+c$ لأن $a$ و $b$ مستقلان عن بعضهما هنا!)**
* **مثال 2 (PDF ص 20 — انتبه لخدعة إعادة الترتيب!):** حل $x\frac{\partial z}{\partial x} + y\frac{\partial z}{\partial y} = z - 5\frac{\partial z}{\partial x} + \frac{\partial z}{\partial x}\frac{\partial z}{\partial y}$:
  * نكتبها برموز مونج: $xp + yq = z - 5p + pq$.
  * **نفصل $z$ في طرف بمفرده:** $z = xp + yq + 5p - pq$ (الآن أصبحت على صيغة كليروت تماماً حيث $f(p,q) = 5p - pq$!).
  * نضع $p = a, q = b$ فنحصل على التكامل التام: $\boxed{z = ax + by + 5a - ab}$.

---

#### 3️⃣ الشكل الثالث $f(z, p, q) = 0$ (القسم 1.3.4 — ص 20–22 بالمذكرة / PDF ص 21–23):
عندما تغيب $x$ و $y$ صراحةً عن المعادلة وتظهر $z, p, q$ فقط، نتبع **الخطوات الأربع في المذكرة (PDF ص 21)**:
1. نفرض المتغير المجمع $u = x + ay$ (حيث $a$ ثابت اختياري).
2. باستخدام قاعدة السلسلة ($\frac{\partial u}{\partial x} = 1$ و $\frac{\partial u}{\partial y} = a$) نستبدل:
   $$p = \frac{dz}{du}, \qquad q = a\frac{dz}{du}$$
3. نعوض في المعادلة الأصلية فتتحول إلى معادلة تفاضلية عادية (ODE) من الرتبة الأولى في $z$ و $u$، فنفصل المتغيرات ونكاملها (مع إضافة ثابت التكامل $c$).
4. نعيد استبدال $u$ بـ $x + ay$.
* **مثال 1 (PDF ص 21–22):** حل $z = p + q$:
  * نضع $u = x + ay$ ونعوض عن $p = \frac{dz}{du}$ و $q = a\frac{dz}{du}$:
    $$z = \frac{dz}{du} + a\frac{dz}{du} = (1 + a)\frac{dz}{du} \implies du = (1 + a)\frac{dz}{z}$$
  * بالتكامل: $u + c = (1 + a)\ln z \implies x + ay + c = \ln\left(z^{1+a}\right) \implies \boxed{z = e^{\frac{x + ay + c}{1 + a}}}$.
* **مثال 2 (PDF ص 22–23):** حل $p^2 z - q^2 = 1$:
  * نعوض بـ $p = \frac{dz}{du}, q = a\frac{dz}{du}$:
    $$\left(\frac{dz}{du}\right)^2 z - a^2\left(\frac{dz}{du}\right)^2 = 1 \implies (z - a^2)\left(\frac{dz}{du}\right)^2 = 1 \implies \pm\sqrt{z - a^2}\,dz = du$$
  * بالتكامل والتعويض عن $u = x + ay$: $\boxed{\pm\frac{2}{3}(z - a^2)^{3/2} = x + ay + c}$.

---

#### 4️⃣ الشكل الرابع القابل للفصل $f_1(x, p) = f_2(y, q)$ (القسم 1.3.5 — ص 22–24 بالمذكرة / PDF ص 23–25):
عندما لا يظهر $z$ في المعادلة، ونستطيع عزل كل ما يخص $(x, p)$ في طرف وكل ما يخص $(y, q)$ في الطرف الآخر:
1. نساوي الطرفين بثابت اختياري $a$: $f_1(x, p) = f_2(y, q) = a$.
2. نحل المعادلة الأولى لإيجاد $p = g_1(x, a)$، ونحل الثانية لإيجاد $q = g_2(y, a)$.
3. نعوض في التفاضل الكلي $dz = p\,dx + q\,dy$ ونكامل الطرفين:
   $$\boxed{z = \int g_1(x, a)\,dx + \int g_2(y, a)\,dy + b}$$
* **مثال 1 (PDF ص 23–24):** حل $p = 2xq^2$:
  * بقسمة الطرفين على $x$ نفصل المتغيرات: $\frac{p}{x} = 2q^2 = a$.
  * إذن: $\frac{p}{x} = a \implies p = ax$، و $2q^2 = a \implies q = \pm\sqrt{\frac{a}{2}}$.
  * نعوض في $dz = p\,dx + q\,dy = ax\,dx \pm \sqrt{\frac{a}{2}}\,dy \xrightarrow{\int} \boxed{z = \frac{a}{2}x^2 \pm \sqrt{\frac{a}{2}}\,y + b}$.

---

### 🏛️ خامساً: طريقة شاربيت العامة (Charpit's Method — القسم 1.3.6، ص 24–26 / PDF ص 25–27)

عندما تحتوي المعادلة $f(x, y, z, p, q) = 0$ على $z$ مع $x, y$ ولا تنطبق عليها الأشكال الأربعة السابقة، نستخدم **معادلات شاربيت المساعدة (PDF ص 25)**:
$$\frac{dp}{f_x + p f_z} = \frac{dq}{f_y + q f_z} = \frac{dz}{-p f_p - q f_q} = \frac{dx}{-f_p} = \frac{dy}{-f_q}$$

#### 📖 المثال المحلول الكامل بطريقة شاربيت (Example 2 — ص 25–26 بالمذكرة / PDF ص 26–27):
**المسألة:** حل المعادلة التفاضلية الجزئية غير الخطية بطريقة شاربيت:
$$2zx - px^2 - 2qxy + pq = 0$$

* **الخطوة 1:** نصفر الطرف الأيمن ونعرف الدالة:
  $$f(x, y, z, p, q) = 2zx - px^2 - 2qxy + pq = 0 \qquad \cdots (5)$$
* **الخطوة 2:** نحسب المشتقات الجزئية الخمس للدالة $f$:
  $$f_x = 2z - 2px - 2qy, \qquad f_y = -2qx, \qquad f_z = 2x, \qquad f_p = -x^2 + q, \qquad f_q = -2xy + p$$
  > ⚠️ **تنبيه خطأ مطبعي في مذكرة الكلية (PDF صفحة 27، السطر الأول):**
  > في أعلى صفحة 27 من ملف الـ PDF، التصقت معادلتا $f_z$ و $f_p$ ببعضهما بدون فاصلة فظهرتا هكذا: `$f_z = 2x f_p = -x^2 + q$`! والصواب أنهما مشتقتان منفصلتان: $f_z = 2x$ ، و $f_p = -x^2 + q$.

* **الخطوة 3:** نعوض في معادلات شاربيت المساعدة:
  $$\frac{dp}{(2z - 2px - 2qy) + p(2x)} = \frac{dq}{-2qx + q(2x)} = \frac{dz}{-p(-x^2+q) - q(-2xy+p)} = \frac{dx}{x^2 - q} = \frac{dy}{2xy - p}$$
* **الخطوة 4 (اختيار أسهل كسر):** انظر إلى مقام الكسر الثاني تحت $dq$:
  $$f_y + q f_z = -2qx + 2qx = 0 \implies \frac{dq}{0} \implies dq = 0 \implies \boxed{q = c} \quad (\text{حيث } c \text{ ثابت اختياري!})$$
* **الخطوة 5 (إيجاد $p$ بالتعويض عن $q = c$ في المعادلة الأصلية 5):**
  $$2zx - px^2 - 2cxy + cp = 0 \implies p(x^2 - c) = 2x(z - cy) \implies \boxed{p = \frac{2x(z - cy)}{x^2 - c}}$$
* **الخطوة 6 (التعويض في التفاضل الكلي $dz = p\,dx + q\,dy$ والتكامل):**
  $$dz = \frac{2x(z - cy)}{x^2 - c}dx + c\,dy \implies dz - c\,dy = \frac{2x(z - cy)}{x^2 - c}dx$$
  لاحظ أن البسط على اليسار $dz - c\,dy$ هو تفاضل المقدار $(z - cy)$ لأن $c$ ثابت! نفصل المتغيرات:
  $$\frac{d(z - cy)}{z - cy} = \frac{2x\,dx}{x^2 - c} \xrightarrow{\text{بتكامل الطرفين}} \ln|z - cy| = \ln|x^2 - c| + \ln b$$
  $$z - cy = b(x^2 - c) \implies \boxed{z = b(x^2 - c) + cy}$$
  وهو التكامل التام المحتوي على الثابتين الاختياريين $b$ و $c$!

</div>


<div dir="rtl" style="text-align:right">

---
## 3. 📝 تحدي الورقة والقلم 1 (صيغة كليروت — المثال 1 ص 19 / PDF ص 20)

**المسألة:** أوجد التكامل التام لمعادلة كليروت التالية:
$$z = p x + q y + p q$$

**خطوات الحل على ورقتك:**
1. تحقق أن المعادلة على صيغة كليروت $z = px + qy + f(p, q)$ حيث $f(p, q) = pq$.
2. عوض عن الميلين بالثابتين الاختياريين $p = a$ و $q = b$.

<details>
<summary>💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b></summary>

استبدل كل $p$ بـ $a$ وكل $q$ بـ $b$ في المعادلة $z = px + qy + pq$. لا تضف أي ثوابت إضافية لأن $(a, b)$ هما الثابتان الاختياريان بالفعل.
</details>

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_clairaut = "z = ax + by + ab"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab3.check_paper_challenge_1(student_clairaut)


<div dir="rtl" style="text-align:right">

---
## 4. 🚀 تحدي جسر مركبة المريخ (من المستويات إلى مخروط المسافة الإقليدية)

نريد الآن حل معادلة المسافة (Eikonal Equation) التي يعتمد عليها رادار المركبة وتوسيع العوائق:
$$p^2 + q^2 = 1$$

**كيف يرتبط التكامل التام (Form 1) بمخروط المسافة الدائري حول الصخرة؟**
1. المعادلة من **الشكل الأول (Form 1: $f(p,q)=0$)** لأنها تحتوي على $p$ و $q$ فقط.
2. إذا وضعنا $p = \cos\alpha$، فإن $q = \sin\alpha$ (لأن $\cos^2\alpha + \sin^2\alpha = 1$).
3. إذن عائلة المستويات المارة بنقطة الأصل هي:
   $$z = x\cos\alpha + y\sin\alpha$$
   كل قيمة للزاوية $\alpha$ تمثل موجة مستوية تسير في اتجاه الزاوية $\alpha$.
4. لإيجاد **الغلاف الهندسي (Envelope)** لجميع الاتجاهات الممكنة $\alpha \in [0, 2\pi)$، نشتق بالنسبة للبارامتر $\alpha$ ونساوي المشتقة بالصفر:
   $$\frac{\partial z}{\partial \alpha} = -x\sin\alpha + y\cos\alpha = 0$$
5. بتربيع المعادلتين وجمعهما للتخلص من $\alpha$:
   $$z^2 + 0^2 = (x\cos\alpha + y\sin\alpha)^2 + (-x\sin\alpha + y\cos\alpha)^2 = x^2(\cos^2\alpha + \sin^2\alpha) + y^2(\sin^2\alpha + \cos^2\alpha)$$

أكمل تبسيط الطرف الأيمن واكتب معادلة سطح المسافة الناتج بدلالة $x, y, z$:

<details>
<summary>💡 <b>تلميح 1 (المتطابقة المثلثية)</b></summary>

بما أن $\cos^2\alpha + \sin^2\alpha = 1$، يتبسط الطرف الأيمن إلى مجموع مربعي الإحداثيين $x$ و $y$.
</details>

<details>
<summary>💡 <b>تلميح 2 (الصيغة النهائية)</b></summary>

المعادلة تربط بين $x^2 + y^2$ و $z^2$، وتمثل مخروط المسافة الذي مقطعه الأفقي عند أي مسافة $z = R$ هو دائرة نصف قطرها $R$.
</details>

</div>


In [ ]:
# خلية تقييم: مخروط المسافة الإقليدية
### START CODE HERE ###
student_envelope = "x^2 + y^2 = z^2"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab3.check_paper_challenge_2(student_envelope)


<div dir="rtl" style="text-align:right">

---
## 5. 💻 التطبيق البرمجي ببايثون: بناء خريطة المسافات وتوسيع العوائق (`terrain.dilate`)

من المعادلة $x^2 + y^2 = z^2$ (أو بالنسبة لمصدر عند $(y_0, x_0)$: $(x - x_0)^2 + (y - y_0)^2 = T^2$)، نحصل على دالة المسافة الإقليدية:
$$T(x, y) = \sqrt{(x - x_0)^2 + (y - y_0)^2}$$

في بايثون، الدالة `np.hypot(dx, dy)` تحسب الجذر التربيعي $\sqrt{\Delta x^2 + \Delta y^2}$ بدقة عددية عالية، وهي نفس القاعدة الهندسية التي تستخدمها دالة [`terrain.dilate(raw, radius)`](rover_path_finder/terrain.py) لإنشاء قرص الأمان الدائري حول كل صخرة!

أكمل الدالة `compute_eikonal_wavefront` أدناه:

</div>


In [ ]:
# دالة تقييم برمجية: compute_eikonal_wavefront
def compute_eikonal_wavefront(grid_size=(50, 50), source=(25, 25)):
    # تحسب خريطة المسافة الإقليدية T(x, y) التي تحقق معادلة Eikonal: p^2 + q^2 = 1
    # ملاحظة: source = (row_y, col_x)
    Ny, Nx = grid_size
    Y, X = np.ogrid[:Ny, :Nx]
    
    ### START CODE HERE ###
    dist = np.hypot(X - source[1], Y - source[0])
    ### END CODE HERE ###
    
    return dist

# تشغيل الاختبار الآلي:
test_lab3.test_eikonal_wavefront_distance(compute_eikonal_wavefront)


<div dir="rtl" style="text-align:right">

---
## 6. 🎨 الرؤية البصرية والربط المباشر بملف `rover_path_finder/terrain.py`
شغّل الخلية التالية لرؤية:
1. **على اليسار:** خريطة المسافة الإقليدية $T(x, y)$ التي تحقق $\|\nabla T\|^2 = p^2 + q^2 = 1$.
2. **على اليمين:** التطبيق الفعلي في المشروع عبر استدعاء دالة `terrain.dilate()` من [`rover_path_finder/terrain.py`](rover_path_finder/terrain.py) لإضافة هامش أمان دائري حول الحفر والصخور قبل تخطيط مسار المركبة!

</div>


In [ ]:
from rover_path_finder import terrain

wavefront = compute_eikonal_wavefront((60, 60), source=(30, 30))
t_data = terrain.make_terrain(N=100, seed=42, rover_radius=3)
elev, raw_hazard, buffered_hazard = t_data["h"], t_data["raw"], t_data["hazard"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

cs = axes[0].contourf(wavefront, levels=20, cmap='plasma')
plt.colorbar(cs, ax=axes[0], label="Euclidean Distance T(x, y)")
axes[0].plot(30, 30, 'wo', markersize=9, label="Source (x0, y0)")
axes[0].set_title(r"1. Eikonal Distance Field: $\|\nabla T\|^2 = p^2 + q^2 = 1$", fontsize=12)
axes[0].legend()

# عرض العوائق الأصلية (بالأحمر الداكن) وهامش الأمان الإقليدي المضاف (بالبرتقالي)
safety_margin_only = buffered_hazard & (~raw_hazard)
overlay = np.zeros((100, 100, 3))
overlay[ ~buffered_hazard ] = [0.95, 0.95, 0.95]   # أرض آمنة
overlay[ safety_margin_only ] = [1.0, 0.65, 0.1]   # هامش الأمان الدائري (r <= 3)
overlay[ raw_hazard ] = [0.75, 0.1, 0.1]           # الصخور والحفر الأصلية

axes[1].imshow(overlay, origin='upper')
axes[1].set_title("2. Real Engine: terrain.dilate(raw, rover_radius=3)", fontsize=12)
axes[1].set_xlabel("x (pixels)")
axes[1].set_ylabel("y (pixels)")

plt.tight_layout()
plt.show()

print("📌 ملخص اللاب 3:")
print("  1) المعادلات غير الخطية من الرتبة الأولى تُحل بالأشكال القياسية الأربعة أو بطريقة شاربيت العامة.")
print("  2) معادلة المسافة p^2 + q^2 = 1 تنتج مخروط المسافة الدائري x^2 + y^2 = z^2.")
print("  3) دالة terrain.dilate() في المشروع تستخدم هذه المسافة الإقليدية لتوسيع العوائق بنصف قطر المركبة.")
